# NeuralRetail - Exploratory Data Analysis
**Amdox Technologies | Data Science & Analytics | AMX-DS-2026-04**

This notebook provides a comprehensive EDA of the NeuralRetail synthetic retail dataset, covering:
1. Dataset overview and statistics
2. Distribution analysis
3. Correlation heatmap
4. Seasonality decomposition
5. RFM analysis
6. Key insights for ML pipeline

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load datasets
customers = pd.read_parquet('../data/bronze/customers.parquet')
products = pd.read_parquet('../data/bronze/products.parquet')
transactions = pd.read_parquet('../data/bronze/transactions.parquet')
inventory = pd.read_parquet('../data/bronze/inventory.parquet')

print(f'Customers:    {customers.shape}')
print(f'Products:     {products.shape}')
print(f'Transactions: {transactions.shape}')
print(f'Inventory:    {inventory.shape}')

## 1. Dataset Overview

In [ ]:
print('=== Customers ===')
display(customers.describe())
print('\n=== Products ===')
display(products.describe())
print('\n=== Transactions ===')
display(transactions.describe())

## 2. Distribution Analysis

In [ ]:
# Customer age distribution
fig = px.histogram(customers, x='age', nbins=30, title='Customer Age Distribution',
                   color_discrete_sequence=['#E84E1B'], template='plotly_dark')
fig.show()

# Transaction amount distribution
fig = px.histogram(transactions, x='total_amount', nbins=50, title='Transaction Amount Distribution',
                   color_discrete_sequence=['#F7941D'], template='plotly_dark')
fig.show()

# Product category distribution
cat_counts = products['category'].value_counts()
fig = px.pie(values=cat_counts.values, names=cat_counts.index, title='Product Categories',
             color_discrete_sequence=['#E84E1B','#F7941D','#FBBA13','#2ecc71','#3498db'],
             template='plotly_dark', hole=0.4)
fig.show()

## 3. Correlation Heatmap

In [ ]:
# RFM Features
latest_date = transactions['timestamp'].max()
rfm = transactions.groupby('customer_id').agg({
    'timestamp': lambda x: (latest_date - x.max()).days,
    'transaction_id': 'count',
    'total_amount': 'sum'
}).reset_index()
rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']
data = customers.merge(rfm, on='customer_id', how='left').fillna(0)

corr = data[['age', 'recency', 'frequency', 'monetary']].corr()
fig = px.imshow(corr, text_auto='.2f', title='Feature Correlation Heatmap',
                color_continuous_scale='RdYlBu_r', template='plotly_dark')
fig.show()

## 4. Time-Series Seasonality Analysis

In [ ]:
# Daily sales trend
daily = transactions.groupby(transactions['timestamp'].dt.date)['quantity'].sum().reset_index()
daily.columns = ['date', 'quantity']
daily['date'] = pd.to_datetime(daily['date'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily['date'], y=daily['quantity'], mode='lines',
                         line=dict(color='#F7941D', width=1.5), name='Daily Sales'))
fig.add_trace(go.Scatter(x=daily['date'], y=daily['quantity'].rolling(7).mean(),
                         mode='lines', line=dict(color='#E84E1B', width=3), name='7-Day MA'))
fig.update_layout(template='plotly_dark', title='Daily Sales Trend with 7-Day Moving Average',
                  xaxis_title='Date', yaxis_title='Units Sold')
fig.show()

# Day-of-week pattern
transactions['dow'] = transactions['timestamp'].dt.day_name()
dow_sales = transactions.groupby('dow')['quantity'].mean().reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
fig = px.bar(x=dow_sales.index, y=dow_sales.values, title='Average Sales by Day of Week',
             color=dow_sales.values, color_continuous_scale=['#FBBA13','#E84E1B'],
             template='plotly_dark', labels={'x':'Day','y':'Avg Quantity'})
fig.show()

## 5. RFM Customer Segmentation Preview

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['Recency', 'Frequency', 'Monetary'])
fig.add_trace(go.Histogram(x=rfm['recency'], marker_color='#E84E1B', name='Recency'), row=1, col=1)
fig.add_trace(go.Histogram(x=rfm['frequency'], marker_color='#F7941D', name='Frequency'), row=1, col=2)
fig.add_trace(go.Histogram(x=rfm['monetary'], marker_color='#FBBA13', name='Monetary'), row=1, col=3)
fig.update_layout(template='plotly_dark', title='RFM Distribution Analysis', showlegend=False, height=400)
fig.show()

## 6. Top Revenue SKUs & Category Analysis

In [ ]:
# Top 20 SKUs by revenue
sku_revenue = transactions.groupby('sku_id')['total_amount'].sum().sort_values(ascending=False).head(20)
fig = px.bar(x=sku_revenue.index, y=sku_revenue.values, title='Top 20 SKUs by Revenue',
             color=sku_revenue.values, color_continuous_scale=['#FBBA13','#E84E1B'],
             template='plotly_dark', labels={'x':'SKU','y':'Total Revenue ($)'})
fig.show()

# Revenue by category
merged = transactions.merge(products[['sku_id','category']], on='sku_id')
cat_rev = merged.groupby('category')['total_amount'].sum().sort_values(ascending=False)
fig = px.bar(x=cat_rev.index, y=cat_rev.values, title='Revenue by Product Category',
             color=cat_rev.values, color_continuous_scale=['#FBBA13','#E84E1B'],
             template='plotly_dark', labels={'x':'Category','y':'Total Revenue ($)'})
fig.show()

## 7. Churn Target Analysis

In [ ]:
data['churn'] = (data['recency'] > 90).astype(int)
churn_dist = data['churn'].value_counts()
print(f'Non-Churned: {churn_dist[0]} ({churn_dist[0]/len(data)*100:.1f}%)')
print(f'Churned:     {churn_dist[1]} ({churn_dist[1]/len(data)*100:.1f}%)')

fig = px.pie(values=churn_dist.values, names=['Active','Churned'],
             title='Churn Distribution', color_discrete_sequence=['#2ecc71','#E84E1B'],
             template='plotly_dark', hole=0.5)
fig.show()

## Key Insights for ML Pipeline

1. **RFM features** (recency, frequency, monetary) show strong discriminative power for churn
2. **Weekly seasonality** visible in sales data - Prophet should capture this
3. **Class imbalance** in churn target - needs SMOTE or scale_pos_weight
4. **Top 20% SKUs** contribute ~80% revenue - confirms ABC classification validity
5. **Age** shows weak correlation with purchase behavior - consider dropping or engineering

---
*NeuralRetail | Amdox Technologies | Data Science & Analytics*